In [ ]:
# from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import TextLoader
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.documents import Document

from langchain_chroma import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
load_dotenv()

In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
GROQ_API_KEY

In [ ]:
llm = ChatGroq(  
model="llama-3.1-8b-instant",  
temperature=0.0,  
max_retries=2, 
)

In [ ]:
llm.invoke("what is mcp in 100 words")

In [ ]:
folder_path="./data/policies"

#### Docs Loading

In [41]:
# """
# 1. retrieve_hr_policy
# 2. retrieve_travel_policy
# 3. retrieve_reimbursement_policy
# 4. retrieve_it_security_policy
# 5. retrieve_ai_usage_policy
# 6. grade_context
# 7. rewrite_query
# 8. generate_grounded_answer
# 9. review_answer_grounding
# 10. ask_clarification
# """

#### Doc Loading

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_documents(folder_path="./data/policies"):
    docs = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".md"):   # adjust extension if needed
            file_path = os.path.join(folder_path, filename)
            loader = TextLoader(file_path, encoding="utf-8")
            loaded_docs = loader.load()
            # normalize metadata for each doc
            for i, doc in enumerate(loaded_docs, start=1):
                source_path = Path(file_path)
                doc.metadata = {
                    "source_file": source_path.name,
                    "policy_domain": source_path.stem,
                    "page_number": str(i),
                }
                docs.append(doc)
    return docs


In [ ]:
docs = load_documents("./data/policies")

#### Chunking

In [ ]:
def split_and_set_metadata(docs, chunk_size=800, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunk_documents = text_splitter.split_documents(docs)

    chunks = []
    for i, chunk_document in enumerate(chunk_documents, start=1):
        metadata = dict(chunk_document.metadata)
        metadata["chunk_id"] = f"chunk_{i}"
        chunks.append(Document(page_content=chunk_document.page_content, metadata=metadata))
    return chunks


In [ ]:

# view

chunks = split_and_set_metadata(docs)

print(f"Loaded {len(docs)} docs.")
print(f"Total chunks: {len(chunks)}")
print(chunks[0].metadata)

In [ ]:
if docs:
    print(f"Loaded {len(docs)} pages from PDF.\n")
    print("First page content:\n", docs[0].page_content[:500], "...")
    print("\nMetadata:", docs[3].metadata)
    print("\nMetadata keys:", docs[0].metadata.keys())

In [ ]:
persist_directory="./vector_store"
chunk_collection_name="policy-chunks"

In [ ]:
import chromadb

In [ ]:
chromadb

In [ ]:
chromadb_client = chromadb.PersistentClient(
    path=persist_directory
)

In [ ]:
chromadb_client

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
embedding_model

In [ ]:

vector_db = Chroma(collection_name=chunk_collection_name,
                   collection_metadata={"hnsw:space": "cosine"}, 
                   embedding_function=embedding_model,
                   client=chromadb_client, 
                   persist_directory=persist_directory
                   )

In [ ]:
# def add_chunks_to_vector_db(chunks, vector_db):
vector_db._collection.count()

vector_db.add_documents(
        documents=chunks,
        ids=[chunk.metadata["chunk_id"] for chunk in chunks],
        )
        

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
# Test retrieval
query = "What are the key financial highlights of Amazon in 2025?"
retrieved_chunks = retriever.invoke(query)

In [ ]:
context = "\n\n".join([
    f"Source: {chunk.metadata.get('source_file')} | "
    f"Policy: {chunk.metadata.get('policy_domain')} | "
    f"Page: {chunk.metadata.get('page_number')} | "
    f"Chunk: {chunk.metadata.get('chunk_id')}\n"
    f"{chunk.page_content}"
    for chunk in retrieved_chunks
])


In [ ]:
print(context)


In [ ]:
system_prompt = """

You are an enterprise policy assistant agent.
Use the Correct tool to Answer the user question.
Rules:
- Do not use outside knowledge.
- If the answer is not available in the context, say: "I could not find this in the provided documents."
- Cite the source file and page number or chunk ID for each key claim.
- Do not invent numbers, dates, risks, or business conclusions.
- Keep the answer clear and business-friendly.

Your Execution architecture:
Select Tool
    |
Retrieve Policy Context
    |
Grade Context: if the context is irrelevant call the required tool
    |
Generate Final Answer

NOTE:
Do not rewrite the query more than once if the context is not matched.

Question:
{user_query}

Retrieved Context:
{context}


Return:
1. Answer
2. Supporting Evidence
3. Sources
4. Confidence: High / Medium / Low

"""

In [ ]:
from langchain.tools.retriever import create_retriever_tool

In [ ]:
retrieve_hr_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_hr_policy",
    description="Search and return information about hr policy."
)

In [ ]:
retrieve_travel_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_travel_policy",
    description="Search and return information about travel policy."
)

In [ ]:
retrieve_reimbursement_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_reimbursement_policy",
    description="Search and return information about reimbursement policy."
)

In [ ]:
retrieve_it_security_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_it_security_policy",
    description="Search and return information about IT security policy."
)

In [ ]:
retrieve_ai_usage_policy = create_retriever_tool(
    retriever=retriever,
    name="retrieve_ai_usage_policy",
    description="Search and return information about ai usage policy."
)

In [ ]:

def grade_retrieved_context(context, user_query):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        Retrived Context (context): The context rerieved by retriever tool.

    Returns:
        str: A decision for whether the documents are relevant or not
    """
    print("---CHECK RELEVANCE---")


In [ ]:
def rewrite_query(grader_output, user_query):
    """
    Transform the user query to produce a better question.

    Args:
        Grader's output: 

    Returns:
        dict: The updated state with re-phrased question
    """
    

In [ ]:
tools = [retrieve_ai_usage_policy, retrieve_it_security_policy, retrieve_reimbursement_policy, retrieve_travel_policy, retrieve_hr_policy, grade_retrieved_context]

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

In [ ]:
agent.invoke(". What should I do if the policy does not mention my scenario?")

In [ ]:
the assignment instruction pdf is attached in this folder. use this for reference - remeber i am proceeding with the  LangGraph Pre-built ReAct Agent without much complex code refer to the demo.ipynb file. just migrate teh code from the .ipynb file to the required files in this directory with only changes that are very much required but in simple logic - do not make the code complex and just correct the logic and the prompt where ever needed do not run the code. : final output: the agent should use the retrieval tools for related context as per the tools already defined in demo.ipynb file- tell me plan before proceeding to the final output